In [3]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import db_dtypes
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '../../BQ.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [20]:
# original code
sql = r"""
DECLARE dynamic_sql STRING;

SET dynamic_sql = (
  SELECT STRING_AGG(
    FORMAT(""""""
      SELECT
        EXTRACT(MONTH FROM `Time`) AS _month,
        REGEXP_EXTRACT(EventLabel, r'^(.+?\\.tab)') AS _EventLabel,
        DeviceId
      FROM `openrice-production.ORGA.SV_%s`
      WHERE platform IN ('android','ios')
        AND region = 'HK'
        AND (EventLabel LIKE '%%HK.LMS2.21.tab%%'
             OR EventLabel LIKE '%%HK.LMS2.9313.tab%%'
        )
    """""", FORMAT_DATE('%Y%m%d', d)),
    "" UNION ALL ""
  )
  FROM UNNEST(GENERATE_DATE_ARRAY(DATE '2026-09-01', DATE '2026-09-30')) AS d
);

EXECUTE IMMEDIATE FORMAT(""""""
  SELECT
    _month, _EventLabel, 
    count(1) as cnt, 
    COUNT(DISTINCT DeviceId) AS device_cnt
  FROM (%s)
  GROUP BY _month, _EventLabel
"""""", dynamic_sql);
"""



In [24]:
sql = r"""
DECLARE dynamic_sql STRING;

SET dynamic_sql = (
  SELECT STRING_AGG(
    FORMAT('''
      SELECT
        EXTRACT(MONTH FROM `Time`) AS _month,
        REGEXP_EXTRACT(EventLabel, r'^(.+?[.]tab)') AS _EventLabel,
        DeviceId
      FROM `openrice-production.ORGA.SV_%s`
      WHERE platform IN ('android', 'ios')
        AND region = 'HK'
        AND (
          EventLabel LIKE '%%HK.LMS2.21.tab%%'
          OR EventLabel LIKE '%%HK.LMS2.9313.tab%%'
        )
    ''', FORMAT_DATE('%Y%m%d', d)),
    ' UNION ALL '
  )
  FROM UNNEST(
    GENERATE_DATE_ARRAY(DATE '2026-09-01', DATE '2026-09-30')
  ) AS d
);

EXECUTE IMMEDIATE FORMAT('''
  SELECT
    _month,
    _EventLabel,
    COUNT(*) AS cnt,
    COUNT(DISTINCT DeviceId) AS device_cnt
  FROM (%s)
  GROUP BY _month, _EventLabel
  ORDER BY _month, _EventLabel
''', dynamic_sql);
"""

In [25]:
df_lms = client.query(sql).to_dataframe()
df_lms

BadRequest: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/openrice-production/queries/afc096de-a3ef-48ed-90af-9a1d262b7878?maxResults=0&location=US&prettyPrint=false: Not found: Table openrice-production:ORGA.SV_20260929 was not found in location US at [2:3]

Location: US
Job ID: afc096de-a3ef-48ed-90af-9a1d262b7878
